In [92]:
# Import des liberairies

#----Python
import os

# ----- Spark
from pyspark.sql.session import SparkSession
import pyspark.sql.functions as sf
from pyspark.ml.feature import StringIndexer, VectorAssembler, OneHotEncoder
from pyspark.ml.pipeline import Pipeline
from pyspark.ml.regression import LinearRegression, RandomForestRegressor, GBTRegressor
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

#----Autres-------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --- --Initialisation Spark ---
spark = (
    SparkSession.builder.appName('Projet')
    .config('spark.driver.memory', '4g') # autoriser le driver d'utiliser jusqu'à 4gb
    .config('spark.executer.memory','4g')
    .config('spark.sql.shuffle.partitions','50') # par defaut c'est 200, 50 pour réduire la tension sur la machine
    .config('spark.sql.execution.arrow.pyspark.enabaled' ,'true') # pour une transition rapide entre pandas et spark df
    .getOrCreate()
)

# --- Formatage du texte ---
# character Gras
B = '\033[1m' 
BE = '\033[0m'

#Chemin vers les données sur ma machine
CHEMIN_INTERIM = '../data/interim'

In [94]:
df_LabelConf = spark.read.parquet(CHEMIN_INTERIM + '/df_LabelConf.parquet')
df_LabelConf.printSchema()

root
 |-- CdSta: string (nullable = true)
 |-- NmNoSta: string (nullable = true)
 |-- DtPrel: string (nullable = true)
 |-- AnPrel: string (nullable = true)
 |-- MoPrel: string (nullable = true)
 |-- NoDep: string (nullable = true)
 |-- CdTpSta: string (nullable = true)
 |-- LbTpSta: string (nullable = true)
 |-- CapNomSta: double (nullable = true)
 |-- A3_Temperature: double (nullable = true)
 |-- A3_pH: double (nullable = true)
 |-- A3_MES: double (nullable = true)
 |-- A3_DBO5: double (nullable = true)
 |-- A3_DCO: double (nullable = true)
 |-- A3_NTK: double (nullable = true)
 |-- A3_NH4: double (nullable = true)
 |-- A3_NO2: double (nullable = true)
 |-- A3_NO3: double (nullable = true)
 |-- A3_PT: double (nullable = true)
 |-- A3_Nglobal: double (nullable = true)
 |-- A3_Volume: double (nullable = true)
 |-- A3_Pluie: double (nullable = true)
 |-- A4_Temperature: double (nullable = true)
 |-- A4_pH: double (nullable = true)
 |-- A4_MES: double (nullable = true)
 |-- A4_DBO5: doub

In [96]:
# On filtre les lignes où on a les entrées et la sorties de rendemment (pour avoir un rendement valide)
df_PreReg = (
    df_LabelConf
    .select(  
        
        # Identifiants (pas pour le modèle, mais pratiques pour debug/analyse)
        'CdSta'
        ,'DtPrel'
        
        # Features catégorielles (à encoder ensuite)
        ,'CatSta'
        ,'NoDep'
        ,'MoPrel'
        ,'AnPrel'
        
        # Features numériques (entrée + station)
        ,'A3_DBO5'
        ,'A3_DCO'
        ,'A3_MES'
        ,'A3_NTK'
        ,'A3_NH4'
        ,'A3_Nglobal'
        ,'CapNomSta'
        
        # Cible
        ,'Rdt_DBO5'
        ,'Rdt_DCO'
        ,'Rdt_MES'
        ,'Rdt_NTK'
        ,'Rdt_NH4'
        ,'Rdt_Nglobal'
    )
    .filter(
        sf.col('Rdt_DBO5').isNotNull()
        & sf.col('Rdt_DCO').isNotNull()
        & sf.col('Rdt_MES').isNotNull()
        & sf.col('Rdt_NTK').isNotNull()
        & sf.col('Rdt_NH4').isNotNull()
        & sf.col('Rdt_Nglobal').isNotNull()
        
        & sf.col('A3_DBO5').isNotNull()
        & sf.col('A3_DCO').isNotNull()
        & sf.col('A3_MES').isNotNull()
        & sf.col('A3_NTK').isNotNull()
        & sf.col('A3_NH4').isNotNull()
        & sf.col('A3_Nglobal').isNotNull()
        & sf.col('CapNomSta').isNotNull()
    )
)
print(df_PreReg.count())
df_PreReg.select('Rdt_DBO5','Rdt_DCO','Rdt_MES','Rdt_NTK','Rdt_NH4','Rdt_Nglobal').describe().show()


73178
+-------+-----------------+-----------------+-----------------+------------------+------------------+------------------+
|summary|         Rdt_DBO5|          Rdt_DCO|          Rdt_MES|           Rdt_NTK|           Rdt_NH4|       Rdt_Nglobal|
+-------+-----------------+-----------------+-----------------+------------------+------------------+------------------+
|  count|            73178|            73178|            73178|             73178|             73178|             73178|
|   mean|97.45244513616875|  94.523235998973|96.47056385149271| 92.88548213628798| 93.82025769677956| 83.99318542664712|
| stddev|6.544023483669362|7.392585801385695|9.627081385162228|12.694240597156162|14.954258790398507|17.348770794394863|
|    min|           -100.0|           -100.0|           -100.0|            -100.0|            -100.0|            -100.0|
|    max|            100.0|            100.0|            100.0|             100.0|             100.0|             100.0|
+-------+-----------------

### String Indexing & One-Hot Encoding

In [98]:
col_idx = ['AnPrel', 'MoPrel', 'NoDep','CatSta']

# Indexing 
indexeur = [StringIndexer(inputCol = c, outputCol = c + '_idx',handleInvalid="keep") for c in col_idx]

# OHE
encodeur= [OneHotEncoder(inputCol = c + '_idx', outputCol = c + '_ohe',handleInvalid="keep") for c in col_idx]

# df_PreRegEncod
df_PreRegEncod = Pipeline(stages=[*indexeur,*encodeur]).fit(df_PreReg).transform(df_PreReg)
df_PreRegEncod.printSchema()

root
 |-- CdSta: string (nullable = true)
 |-- DtPrel: string (nullable = true)
 |-- CatSta: string (nullable = true)
 |-- NoDep: string (nullable = true)
 |-- MoPrel: string (nullable = true)
 |-- AnPrel: string (nullable = true)
 |-- A3_DBO5: double (nullable = true)
 |-- A3_DCO: double (nullable = true)
 |-- A3_MES: double (nullable = true)
 |-- A3_NTK: double (nullable = true)
 |-- A3_NH4: double (nullable = true)
 |-- A3_Nglobal: double (nullable = true)
 |-- CapNomSta: double (nullable = true)
 |-- Rdt_DBO5: double (nullable = true)
 |-- Rdt_DCO: double (nullable = true)
 |-- Rdt_MES: double (nullable = true)
 |-- Rdt_NTK: double (nullable = true)
 |-- Rdt_NH4: double (nullable = true)
 |-- Rdt_Nglobal: double (nullable = true)
 |-- AnPrel_idx: double (nullable = false)
 |-- MoPrel_idx: double (nullable = false)
 |-- NoDep_idx: double (nullable = false)
 |-- CatSta_idx: double (nullable = false)
 |-- AnPrel_ohe: vector (nullable = true)
 |-- MoPrel_ohe: vector (nullable = true)
 

In [100]:
# Features finales
col_feat = ['AnPrel_ohe', 'MoPrel_ohe', 'NoDep_ohe', 'CatSta_ohe','CapNomSta', 'A3_DBO5', 'A3_DCO', 'A3_MES','A3_NTK','A3_NH4','A3_Nglobal']

# Assembleur : création de la colonne feature
assembleur = VectorAssembler(inputCols = col_feat, outputCol= 'features')

# df_Reg
df_Reg = (
    Pipeline(stages=[assembleur])
    .fit(df_PreRegEncod)
    .transform(df_PreRegEncod)
    .select(  
        # Identifiants (pas pour le modèle, mais pratiques pour debug/analyse)
        'CdSta'
        ,'DtPrel'
        
        # Features catégorielles (à encoder ensuite)
        ,'CatSta'
        ,'NoDep'
        ,'MoPrel'
        ,'AnPrel'
        
        # Features numériques (entrée + station)
        ,'A3_DBO5'
        ,'A3_DCO'
        ,'A3_MES'
        ,'A3_NTK'
        ,'A3_NH4'
        ,'A3_Nglobal'
        ,'CapNomSta'
        
        # Cible
        ,'Rdt_DBO5'
        ,'Rdt_DCO'
        ,'Rdt_MES'
        ,'Rdt_NTK'
        ,'Rdt_NH4'
        
        # feature
        ,'features'
    ))
df_Reg.printSchema()

root
 |-- CdSta: string (nullable = true)
 |-- DtPrel: string (nullable = true)
 |-- CatSta: string (nullable = true)
 |-- NoDep: string (nullable = true)
 |-- MoPrel: string (nullable = true)
 |-- AnPrel: string (nullable = true)
 |-- A3_DBO5: double (nullable = true)
 |-- A3_DCO: double (nullable = true)
 |-- A3_MES: double (nullable = true)
 |-- A3_NTK: double (nullable = true)
 |-- A3_NH4: double (nullable = true)
 |-- A3_Nglobal: double (nullable = true)
 |-- CapNomSta: double (nullable = true)
 |-- Rdt_DBO5: double (nullable = true)
 |-- Rdt_DCO: double (nullable = true)
 |-- Rdt_MES: double (nullable = true)
 |-- Rdt_NTK: double (nullable = true)
 |-- Rdt_NH4: double (nullable = true)
 |-- features: vector (nullable = true)



In [102]:
# creation de données d'entrainement et test data
df_train, df_test = df_Reg.randomSplit([.8,.2],seed=2026)

In [104]:
def eval_metrics(reg_method, df_predictions, colCibl,colPred):
    print(f'Evalution des métriques {reg_method} pour {colCibl} :')
    for metric in ['rmse','r2','mae','var']:
        evaluator = RegressionEvaluator(predictionCol = colPred ,labelCol = colCibl ,metricName = metric)
        eval_value = round(evaluator.evaluate(df_predictions),2)
        print(f'{B}{metric}{BE}:{eval_value}')

### Random Forest

In [107]:
col_cibles =['Rdt_DBO5', 'Rdt_DCO', 'Rdt_MES', 'Rdt_NTK', 'Rdt_NH4']
# col_cibles =['Rdt_DBO5', 'Rdt_DCO']

In [109]:
# 1. Create a list of stages (one RF model per target)
stages = []
for colCibl in col_cibles:
    # We give each model a unique prediction column name
    rf = RandomForestRegressor(
        featuresCol='features', 
        labelCol=colCibl, 
        predictionCol=f"pred_{colCibl}", 
        numTrees=50, 
        maxDepth=10, 
        seed=2026
    )
    stages.append(rf)

# 2. Wrap everything into a single Pipeline
pipeline = Pipeline(stages=stages)

# 3. Fit the model once
# This trains all targets in sequence (or parallel if your cluster allows)
multi_target_model = pipeline.fit(df_train)

# 4. Predict everything at once
df_predictions = multi_target_model.transform(df_test)

In [110]:
# Now df_predictions contains: pred_Rdt_DBO5, pred_Rdt_DCO, etc.
df_predictions.select( *[c for c in col_cibles], *[f"pred_{c}" for c in col_cibles]).show(truncate=10)

+----------+----------+----------+----------+----------+-------------+------------+------------+------------+------------+
|  Rdt_DBO5|   Rdt_DCO|   Rdt_MES|   Rdt_NTK|   Rdt_NH4|pred_Rdt_DBO5|pred_Rdt_DCO|pred_Rdt_MES|pred_Rdt_NTK|pred_Rdt_NH4|
+----------+----------+----------+----------+----------+-------------+------------+------------+------------+------------+
|97.6923...|93.0693...|98.4615...|84.3243...|      84.4|   96.8640...|  92.4097...|  95.3019...|  90.9571...|  92.2354...|
|95.3571...|89.2447...|89.0909...|73.4265...|84.0425...|   96.9669...|  95.1654...|  96.8413...|  86.2543...|  80.5725...|
|     99.76|98.6861...|     99.92|90.2857...|85.8407...|   97.0560...|  95.1475...|  97.1617...|  80.8295...|  75.9886...|
|      92.0|74.5810...|85.8823...|38.7755...|27.7793...|   94.9076...|  87.3297...|  91.0383...|  76.5200...|  81.6132...|
|      65.0|58.3554...|65.7142...|17.2413...|       0.0|   92.7510...|  85.2603...|  91.6775...|  69.1844...|  70.1787...|
|97.6470...|96.6

In [119]:
    # Affichage
    print(f'\n{B}Modele Random Forest (RF){BE} de {B}{colCibl} :{BE}')
    for colCibl in col_cibles:
        eval_metrics('RF', df_predictions, colCibl, f"pred_{colCibl}")

    print(f'-------------------------------------------\n-------------------------------------------')


Modele Random Forest (RF) de Rdt_DBO5 :
Evalution des métriques RF pour Rdt_DBO5 :
rmse:6.19
r2:0.26
mae:1.45
var:11.59
Evalution des métriques RF pour Rdt_DCO :
rmse:6.41
r2:0.29
mae:2.17
var:15.82
Evalution des métriques RF pour Rdt_MES :
rmse:9.27
r2:0.19
mae:2.69
var:18.26
Evalution des métriques RF pour Rdt_NTK :
rmse:10.63
r2:0.31
mae:4.77
var:44.62
Evalution des métriques RF pour Rdt_NH4 :
rmse:13.0
r2:0.24
mae:5.84
var:48.63
-------------------------------------------
-------------------------------------------


In [121]:
spark.stop

<bound method SparkSession.stop of <pyspark.sql.session.SparkSession object at 0x000001F2EE5C7820>>